# Практикум: пропуски и дубликаты

Анализ и структуры данных, 9 класс. Раздел 3.2, занятие 4.

Двенадцать задач на трёх наборах. Все файлы лежат рядом с этой тетрадью:

* `titanic.csv` — пассажиры «Титаника», 891 строка, тот же набор, что на слайдах;
* `registrations.csv` — заявки на школьный этап олимпиады, 89 строк. Учебный набор:
  сгенерирован для занятия, персональных данных в нём нет;
* `temperature.csv` — минимальная суточная температура в Мельбурне за 1981–1990 годы,
  3650 дней (Australian Bureau of Meteorology).

Главное правило пары: функция **возвращает новую таблицу или новый столбец** и не меняет то,
что ей передали. Исходные данные неприкосновенны, и тесты это проверяют. Если внутри функции
нужно что-то поменять, начинайте с копии: `result = df.copy()`.

Уровни задач:

* **1–6 — обязательные**, по одной идее на задачу;
* **7–10 — на сочетание идей**;
* **11–12 — со звёздочкой**.

Пишите код внутри функции и запускайте ячейку. Если все `assert` проходят, внизу печатается
«ок». Падение `assert` значит, что какой-то случай не обработан: прочитайте сообщение,
найдите случай и почините.

### Загрузка данных

Выполните эту ячейку первой. Функция `untouched()` проверяет, что исходные таблицы
не изменились, — её вызывают тесты.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

for name in ["titanic.csv", "registrations.csv", "temperature.csv"]:
    assert Path(name).exists(), f"Файл {name} должен лежать рядом с этой тетрадью"

titanic = pd.read_csv("titanic.csv")
reg = pd.read_csv("registrations.csv")
temperature = pd.read_csv("temperature.csv")

# копии, чтобы проверять, что функции не портят исходные таблицы
_COPIES = [(titanic, titanic.copy()), (reg, reg.copy()), (temperature, temperature.copy())]


def untouched():
    """True, если ни одна исходная таблица не изменилась."""
    return all(df.equals(copy) for df, copy in _COPIES)


print("версия pandas:", pd.__version__)
print("titanic:", titanic.shape, "  reg:", reg.shape, "  temperature:", temperature.shape)
titanic.head()

## Часть 1. Обязательные задачи

По одной идее на задачу.

### Задача 1. Скрытые пропуски

В анкете пропуски записаны кто как привык: прочерком, словами «нет данных», пустой строкой.
`isna()` таких не видит — для него это обычные строки. Функция получает столбец и список
таких меток и возвращает **новый** столбец, где каждая метка заменена настоящим пропуском `NaN`.

Остальные значения не меняются, в том числе `"0"`: ноль — это ответ, а не пропуск.

```
answers = pd.Series(["15", "-", "нет данных", "0", "", "17"])
mark_missing(answers, ["-", "нет данных", ""])
->  ["15", NaN, NaN, "0", NaN, "17"]
```

*Подсказка:* `s.isin(markers)` даёт маску «значение — метка». Метод `s.where(условие)`
оставляет значения там, где условие истинно, а в остальных местах ставит `NaN`.

In [ ]:
def mark_missing(s: pd.Series, markers: list) -> pd.Series:
    """Новый столбец: значения из markers заменены на NaN."""
    return s.mask(s.isin(markers))


answers = pd.Series(["15", "-", "нет данных", "0", "", "17"])
backup = answers.copy()
res = mark_missing(answers, ["-", "нет данных", ""])
assert res is not None, "функция ничего не вернула"
assert res.isna().tolist() == [False, True, True, False, True, False], \
    "пропуском становится каждое значение из списка markers, и только оно"
assert res[0] == "15" and res[3] == "0" and res[5] == "17", "остальные значения не должны меняться"
assert answers.equals(backup), "исходный столбец изменился"

# метки бывают числовыми: -999 — частый код «нет данных» в выгрузках
codes = pd.Series([3.5, -999.0, 4.0, -999.0, 0.0])
res2 = mark_missing(codes, [-999.0])
assert res2.isna().sum() == 2
assert res2.sum() == 7.5, "ноль — это значение, а не пропуск"

assert mark_missing(answers, []).isna().sum() == 0, "пустой список меток — ничего не меняется"
assert mark_missing(answers, ["?"]).isna().sum() == 0, "меток нет в данных — ничего не меняется"
assert mark_missing(pd.Series([1.0, np.nan, 2.0]), [2.0]).isna().tolist() == [False, True, True], \
    "настоящие пропуски должны остаться пропусками"
print("Задача 1: ок")

### Задача 2. Почти пустые признаки

Удалите из таблицы столбцы, в которых доля пропусков **строго больше** порога `max_share`
(число от 0 до 1). Строки не трогайте, порядок оставшихся столбцов сохраните.

```
drop_sparse_columns(titanic, 0.5).shape   ->  (891, 14)     # ушёл deck
drop_sparse_columns(titanic, 0.1).shape   ->  (891, 13)     # ушли deck и age
```

*Подсказка:* `df.isna().mean()` — доля пропусков в каждом столбце: среднее из `True` и `False`.
Отберите из этого результата то, что больше порога, и передайте имена в `drop(columns=...)`.

In [ ]:
def drop_sparse_columns(df: pd.DataFrame, max_share: float) -> pd.DataFrame:
    """Таблица без столбцов, где доля пропусков больше max_share."""
    d= df.columns[df.isna().mean() > max_share]
    return df.drop(columns=d)


res = drop_sparse_columns(titanic, 0.5)
assert res is not None, "функция ничего не вернула"
assert list(res.columns) == [c for c in titanic.columns if c != "deck"], "при пороге 0.5 уходит только deck"
assert len(res) == 891, "строки удалять не нужно"

res = drop_sparse_columns(titanic, 0.1)
assert res.shape == (891, 13) and "age" not in res.columns and "deck" not in res.columns

assert drop_sparse_columns(titanic, 1.0).shape == (891, 15), "порог 1 — не удаляется ничего"
assert drop_sparse_columns(titanic, 0.0).shape == (891, 11), "порог 0 — уходят все столбцы с пропусками"

half = pd.DataFrame({"a": [1.0, None], "b": [1, 2]})
assert list(drop_sparse_columns(half, 0.5).columns) == ["a", "b"], \
    "доля 0.5 при пороге 0.5 — не «строго больше», столбец остаётся"
assert untouched(), "исходная таблица изменилась"
print("Задача 2: ок")

### Задача 3. Заполнение медианой

Верните **новую** таблицу, в которой пропуски в столбце `column` заполнены медианой этого
столбца. Остальные столбцы не меняются.

```
filled = fill_median(titanic, "age")
filled["age"].isna().sum()        ->  0
(filled["age"] == 28.0).sum()     ->  202     # 25 настоящих двадцативосьмилетних + 177 заполненных
```

*Подсказка:* `fillna(значение)` возвращает новый столбец, его нужно присвоить
в копию таблицы.

In [ ]:
def fill_median(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Новая таблица: пропуски в column заполнены медианой столбца."""
    res = df.copy()
    res[column] = res[column].fillna(res[column].median())
    return res


filled = fill_median(titanic, "age")
assert filled is not None, "функция ничего не вернула"
assert filled["age"].isna().sum() == 0
assert (filled["age"] == 28.0).sum() == 202, "пропуски заполняются медианой столбца — 28"
assert filled.shape == titanic.shape
assert filled["deck"].isna().sum() == 688, "другие столбцы трогать не нужно"
assert untouched(), "исходная таблица изменилась"

# медиана, а не среднее: здесь они сильно различаются
small = pd.DataFrame({"x": [1.0, 2.0, 3.0, 100.0, None], "y": ["a", None, "b", "c", "d"]})
res = fill_median(small, "x")
assert res["x"].tolist() == [1.0, 2.0, 3.0, 100.0, 2.5], "заполнять нужно медианой (2.5), а не средним (26.5)"
assert res["y"].isna().sum() == 1
assert small["x"].isna().sum() == 1, "исходная таблица изменилась"
print("Задача 3: ок")

### Задача 4. Самое частое значение

Для категориального признака среднего и медианы нет. Заполните пропуски в столбце `column`
**самым частым значением**. Если самых частых несколько, берите первое по алфавиту —
так работает метод `mode()`.

```
fill_mode(titanic, "embark_town")["embark_town"].value_counts()["Southampton"]   ->  646
```

*Подсказка:* `mode()` возвращает не одно значение, а столбец (самых частых может быть
несколько). Нужное — его первый элемент.

In [ ]:
def fill_mode(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Новая таблица: пропуски в column заполнены самым частым значением."""
    res = df.copy()
    res[column] = res[column].fillna(res[column].mode()[0])
    return res


res = fill_mode(titanic, "embark_town")
assert res is not None, "функция ничего не вернула"
assert res["embark_town"].isna().sum() == 0
assert res["embark_town"].value_counts()["Southampton"] == 646
assert res.loc[61, "embark_town"] == "Southampton" and res.loc[829, "embark_town"] == "Southampton"
assert res["embarked"].isna().sum() == 2, "другие столбцы трогать не нужно"
assert untouched(), "исходная таблица изменилась"

tie = pd.DataFrame({"c": ["b", "a", "b", "a", None]})
assert fill_mode(tie, "c")["c"].tolist() == ["b", "a", "b", "a", "a"], \
    "при равных частотах — первое по алфавиту"

nums = pd.DataFrame({"n": [3.0, None, 3.0, 7.0]})
assert fill_mode(nums, "n")["n"].tolist() == [3.0, 3.0, 3.0, 7.0]
print("Задача 4: ок")

### Задача 5. Пропуски внутри ряда

Значения идут по порядку во времени. Пропуск **между** двумя известными значениями заполните
по прямой: два пропуска между 1 и 4 становятся 2 и 3. Пропуски в начале и в конце ряда
оставьте: с одной стороны соседа нет, и любое число там будет выдумкой.

```
fill_inside(pd.Series([1.0, None, None, 4.0]))         ->  [1.0, 2.0, 3.0, 4.0]
fill_inside(pd.Series([None, 2.0, None, 6.0, None]))   ->  [NaN, 2.0, 4.0, 6.0, NaN]
```

*Подсказка:* метод `interpolate()`. Без дополнительного параметра он заполнит хвост ряда
последним значением — проверьте это сами. Нужный параметр найдите в справке:
`help(pd.Series.interpolate)`, параметр `limit_area`.

In [ ]:
def fill_inside(s: pd.Series) -> pd.Series:
    """Новый ряд: пропуски между известными значениями заполнены по прямой, края не тронуты."""
    return s.interpolate(limit_area="inside")

res = fill_inside(pd.Series([1.0, None, None, 4.0]))
assert res is not None, "функция ничего не вернула"
assert res.tolist() == [1.0, 2.0, 3.0, 4.0], "между 1 и 4 — по прямой: 2 и 3"

s = pd.Series([None, 2.0, None, 6.0, None])
backup = s.copy()
res = fill_inside(s)
assert res.iloc[1:4].tolist() == [2.0, 4.0, 6.0]
assert pd.isna(res.iloc[4]), "хвост ряда заполнять нельзя: справа соседа нет"
assert pd.isna(res.iloc[0]), "начало ряда заполнять нельзя: слева соседа нет"
assert s.equals(backup), "исходный ряд изменился"

assert fill_inside(pd.Series([5.0, None, None])).isna().sum() == 2
assert fill_inside(temperature["temp"]).equals(temperature["temp"]), "в ряду без пропусков ничего не меняется"
assert fill_inside(pd.Series([np.nan, np.nan])).isna().sum() == 2
print("Задача 5: ок")

### Задача 6. Сколько лишних копий

Верните число строк, которые **повторяют более раннюю** строку. Если `key` не задан,
строки сравниваются по всем столбцам (полные дубликаты); если задан список столбцов —
только по ним (дубликаты по ключу). Первое вхождение копией не считается.

```
count_copies(reg)                              ->  6
count_copies(reg, ["student_id", "subject"])   ->  15
count_copies(titanic)                          ->  107
```

*Подсказка:* `duplicated` и его параметр `subset`.

In [ ]:
def count_copies(df: pd.DataFrame, key: list = None) -> int:
    """Число строк, повторяющих более раннюю: по всем столбцам или по столбцам key."""
    return int(df.duplicated(subset=key).sum())

res = count_copies(reg)
assert res is not None, "функция ничего не вернула"
assert res == 6, "полных копий в заявках шесть: первое вхождение копией не считается"
assert count_copies(reg, ["student_id", "subject"]) == 15, "по ключу сравниваются только столбцы key"
assert count_copies(reg, ["student_id"]) == 51, "по одному student_id: 89 заявок, 38 разных учеников"
assert count_copies(titanic) == 107

pairs = pd.DataFrame({"a": [1, 1, 1, 2], "b": ["x", "x", "y", "y"]})
assert count_copies(pairs) == 1, "первое вхождение копией не считается"
assert count_copies(pairs, ["a"]) == 2
assert count_copies(pairs.head(0)) == 0
assert untouched(), "исходная таблица изменилась"
print("Задача 6: ок")

## Часть 2. На сочетание идей

### Задача 7. Флаг перед заполнением

После заполнения уже не видно, какие значения настоящие, а какие придуманы. Поэтому перед
заполнением добавляют флаг. Верните новую таблицу, в которой:

* в конце добавлен столбец `<column>_missing`: `True` там, где значение **было** пропущено;
* пропуски в `column` заполнены значением `value`.

```
res = fill_with_flag(titanic, "age", 28.0)
res["age_missing"].sum()     ->  177
res["age"].isna().sum()      ->  0
list(res.columns)[-1]        ->  "age_missing"
```

In [ ]:
def fill_with_flag(df: pd.DataFrame, column: str, value) -> pd.DataFrame:
    """Новая таблица: флаг column_missing и заполнение пропусков значением value."""
    res = df.copy()
    res[column + "_missing"] = res[column].isna()
    res[column] = res[column].fillna(value)
    return res


res = fill_with_flag(titanic, "age", 28.0)
assert res is not None, "функция ничего не вернула"
assert "age_missing" in res.columns
assert res["age_missing"].sum() == 177, "флаг нужно поставить ДО заполнения"
assert res["age"].isna().sum() == 0
assert list(res.columns) == list(titanic.columns) + ["age_missing"]
assert res["age_missing"].dtype == bool, "флаг — столбец из True и False"
assert res.loc[res["age_missing"], "age"].eq(28.0).all()
assert untouched(), "исходная таблица изменилась"

res = fill_with_flag(titanic, "deck", "неизвестно")
assert res["deck_missing"].sum() == 688
assert res["deck"].value_counts()["неизвестно"] == 688

res = fill_with_flag(reg, "grade", "?")
assert res["grade_missing"].sum() == 0 and res.shape == (89, 6), "нет пропусков — флаг весь False"
print("Задача 7: ок")

### Задача 8. Медиана по группе

Общая медиана возраста — 28 лет. Но в первом классе ехали заметно старше, чем в третьем,
и мужчины старше женщин. Заполните пропуски в `column` **медианой группы**, к которой
относится строка. Группа задаётся `by` — названием столбца или списком названий.
Если в группе нет ни одного известного значения, пропуск остаётся.

```
res = fill_group_median(titanic, "age", ["pclass", "sex"])
res.loc[5, "age"]     ->  25.0     # мужчина, 3 класс
res.loc[31, "age"]    ->  35.0     # женщина, 1 класс
```

*Подсказка:* `df.groupby(by)[column].transform("median")` возвращает столбец той же длины,
что таблица: в каждой строке стоит медиана её группы. Этим столбцом и заполняйте:
`fillna` принимает не только число, но и столбец.

In [ ]:
def fill_group_median(df: pd.DataFrame, column: str, by) -> pd.DataFrame:
    """Новая таблица: пропуски в column заполнены медианой группы by."""
    res = df.copy()
    group_median = res.groupby(by)[column].transform("median")
    res[column] = res[column].fillna(group_median)
    return res


res = fill_group_median(titanic, "age", ["pclass", "sex"])
assert res is not None, "функция ничего не вернула"
assert res.loc[5, "age"] == 25.0, "мужчина 3 класса: медиана группы 25, а не общая 28"
assert res.loc[31, "age"] == 35.0
assert res.loc[17, "age"] == 30.0
assert res["age"].isna().sum() == 0
known = titanic["age"].notna()
assert res.loc[known, "age"].equals(titanic.loc[known, "age"]), "известные значения менять нельзя"
assert untouched(), "исходная таблица изменилась"

res = fill_group_median(titanic, "age", "pclass")
assert res.loc[5, "age"] == 24.0 and res.loc[31, "age"] == 37.0

lonely = pd.DataFrame({"g": ["a", "a", "a", "b", "b"], "v": [1.0, None, 5.0, None, None]})
res = fill_group_median(lonely, "v", "g")
assert res["v"].tolist()[:3] == [1.0, 3.0, 5.0]
assert res["v"].isna().sum() == 2, "в группе b нечем заполнять — пропуски остаются"
print("Задача 8: ок")

### Задача 9. Последняя заявка

Ученик мог отправить заявку на один и тот же предмет несколько раз: нажал кнопку дважды,
исправил класс, донёс согласие родителей. Действительна **последняя по времени** заявка
для каждой пары «ученик — предмет». Верните таблицу только с такими заявками,
упорядоченную по времени отправки, с индексом 0, 1, 2, …

```
res = latest_registrations(reg)
len(res)      ->  74
```

*Подсказка:* время записано строкой вида `2026-09-30 16:19:39`. Такие строки при сортировке
встают в правильном порядке — переводить их в даты не нужно. Дальше `drop_duplicates`:
у него есть параметры `subset` и `keep`.

In [ ]:
def latest_registrations(df: pd.DataFrame) -> pd.DataFrame:
    """Только последняя заявка для каждой пары (student_id, subject), по времени."""
    return (df.sort_values("submitted_at").drop_duplicates(subset=["student_id", "subject"], keep="last").reset_index(drop=True))

res = latest_registrations(reg)
assert res is not None, "функция ничего не вернула"
assert len(res) == 74, "по одной заявке на каждую пару ученик — предмет"
assert not res.duplicated(subset=["student_id", "subject"]).any(), "пара ученик — предмет должна быть одна"
assert list(res.index) == list(range(74)), "индекс должен идти 0, 1, 2, ..."
assert res["submitted_at"].is_monotonic_increasing, "упорядочить по времени отправки"


def row(table, student, subject):
    found = table[(table["student_id"] == student) & (table["subject"] == subject)]
    assert len(found) == 1
    return found.iloc[0]


assert row(res, "S102", "математика")["grade"] == "9А", "класс исправлен в последней заявке"
assert row(res, "S120", "физика")["consent"] == "да", "согласие донесено в последней заявке"
assert row(res, "S120", "математика")["consent"] == "да"
assert row(res, "S121", "математика")["grade"] == "11Б"
assert untouched(), "исходная таблица изменилась"

# порядок строк в файле не должен влиять на результат
shuffled = reg.sample(frac=1, random_state=1)
res2 = latest_registrations(shuffled)
assert len(res2) == 74
assert row(res2, "S102", "математика")["grade"] == "9А", "сначала сортировка по времени, потом отбор"
assert row(res2, "S110", "химия")["consent"] == "да"
print("Задача 9: ок")

### Задача 10. Сравнение стратегий

Возьмите столбец с пропусками и обработайте его тремя способами:

* `"удаление"` — выбросить пропуски;
* `"среднее"` — заполнить средним;
* `"медиана"` — заполнить медианой.

Для каждого способа посчитайте среднее и стандартное отклонение (`std()` из pandas)
результата. Верните словарь «способ → (среднее, отклонение)», числа округлите до двух знаков.

```
compare_strategies(titanic["age"])
->  {"удаление": (29.7, 14.53), "среднее": (29.7, 13.0), "медиана": (29.36, 13.02)}
```

Когда тесты пройдут, ответьте в ячейке ниже: какая стратегия не меняет среднее и почему;
почему при заполнении разброс уменьшается.

In [ ]:
def compare_strategies(s: pd.Series) -> dict:
    """Словарь: способ обработки пропусков -> (среднее, стандартное отклонение)."""
    res = {}
    for name, filled in [
        ("удаление", s.dropna()),
        ("среднее", s.fillna(s.mean())),
        ("медиана", s.fillna(s.median())),
    ]:
        res[name] = (round(filled.mean(), 2), round(filled.std(), 2))
    return res

res = compare_strategies(titanic["age"])
assert res is not None, "функция ничего не вернула"
assert res == {"удаление": (29.7, 14.53), "среднее": (29.7, 13.0), "медиана": (29.36, 13.02)}, res

s = pd.Series([1.0, 2.0, 3.0, 10.0, None, None])
backup = s.copy()
assert compare_strategies(s) == {"удаление": (4.0, 4.08), "среднее": (4.0, 3.16), "медиана": (3.5, 3.26)}
assert s.equals(backup), "исходный столбец изменился"

full = pd.Series([2.0, 4.0, 9.0])
r = compare_strategies(full)
assert r["удаление"] == r["среднее"] == r["медиана"] == (5.0, 3.61), "без пропусков стратегии совпадают"
print("Задача 10: ок")

*Ваш ответ:*

## Часть 3. Задачи со звёздочкой

### Задача 11. Интерполяция по группам ★

В длинной таблице подряд идут ряды температур нескольких городов. Заполните пропуски
по прямой **внутри ряда каждого города**: значения соседнего города при этом использовать
нельзя, края ряда не трогать (как в задаче 5).

```
res = fill_inside_groups(CITIES, "temp", "city")
Москва:    [5, 6, 7, 6, 5, 4]
Сочи:      [NaN, 15, 16, 17, 18, NaN]
Мурманск:  [-1, -2, -3, -4, -3, -2]
```

Строки разных городов могут идти вперемешку — порядок дней внутри каждого города
при этом сохраняется.

*Подсказка:* `for name, group in df.groupby(by):` перебирает группы; `group.index` — номера
строк группы в исходной таблице, по ним результат записывается обратно через `loc`.

In [ ]:
CITIES = pd.DataFrame({
    "city": ["Москва"] * 6 + ["Сочи"] * 6 + ["Мурманск"] * 6,
    "day": list(range(1, 7)) * 3,
    "temp": [5, None, 7, None, None, 4,
             None, 15, 16, None, 18, None,
             -1, None, -3, -4, None, -2],
})


def fill_inside_groups(df: pd.DataFrame, column: str, by: str) -> pd.DataFrame:
    """Новая таблица: column заполнен по прямой внутри каждой группы by, края не тронуты."""
    # ваш код здесь
    pass


EXPECTED = {
    "Москва": [5.0, 6.0, 7.0, 6.0, 5.0, 4.0],
    "Сочи": [np.nan, 15.0, 16.0, 17.0, 18.0, np.nan],
    "Мурманск": [-1.0, -2.0, -3.0, -4.0, -3.0, -2.0],
}


def check(res, source):
    assert res is not None, "функция ничего не вернула"
    for city, values in EXPECTED.items():
        got = res.loc[source["city"] == city].sort_values("day")["temp"]
        assert np.allclose(got, values, equal_nan=True), f"{city}: {got.tolist()}"


backup = CITIES.copy()
check(fill_inside_groups(CITIES, "temp", "city"), CITIES)
assert CITIES.equals(backup), "исходная таблица изменилась"

mixed = CITIES.sort_values(["day", "city"])          # города вперемешку
check(fill_inside_groups(mixed, "temp", "city"), mixed)

res = fill_inside_groups(CITIES, "temp", "city")
assert list(res.columns) == ["city", "day", "temp"] and len(res) == 18
print("Задача 11: ок")

### Задача 12. Проверка стратегии на правде ★

Как узнать, какая стратегия заполнения лучше, если правды о пропусках мы не знаем?
Спрятать **известные** значения, заполнить их и сравнить с тем, что было на самом деле.

Функция получает полный ряд `s` без пропусков и список номеров позиций `positions`
(первый и последний элемент среди них не встречаются). Спрячьте значения на этих позициях
и заполните тремя способами:

* `"среднее"` — средним по оставшимся значениям;
* `"предыдущее"` — предыдущим известным значением;
* `"интерполяция"` — по прямой между соседями.

Для каждого способа посчитайте среднюю абсолютную ошибку на спрятанных позициях:
среднее из `|заполненное − настоящее|`. Верните словарь «способ → ошибка», округлив
до двух знаков. Исходный ряд не меняйте.

```
gap_fill_error(temperature["temp"], POSITIONS)
->  {"среднее": 3.43, "предыдущее": 2.04, "интерполяция": 1.57}
```

*Подсказка:* позиции — номера по порядку, значит, нужен `iloc`.

In [ ]:
rng = np.random.default_rng(7)
POSITIONS = sorted(rng.choice(np.arange(1, len(temperature) - 1), 200, replace=False))


def gap_fill_error(s: pd.Series, positions: list) -> dict:
    """Средняя абсолютная ошибка трёх способов заполнения на спрятанных позициях."""
    # ваш код здесь
    pass


backup = temperature["temp"].copy()
res = gap_fill_error(temperature["temp"], POSITIONS)
assert res is not None, "функция ничего не вернула"
assert res == {"среднее": 3.43, "предыдущее": 2.04, "интерполяция": 1.57}, res
assert temperature["temp"].equals(backup) and untouched(), "исходный ряд изменился"

line = pd.Series([10.0, 20.0, 30.0, 40.0, 50.0])
assert gap_fill_error(line, [2]) == {"среднее": 0.0, "предыдущее": 10.0, "интерполяция": 0.0}
assert gap_fill_error(line, [1, 2]) == {"среднее": 8.33, "предыдущее": 15.0, "интерполяция": 0.0}, \
    "среднее считается только по оставшимся значениям"
assert line.tolist() == [10.0, 20.0, 30.0, 40.0, 50.0], "исходный ряд изменился"

ints = pd.Series([1, 5, 9, 2])
assert gap_fill_error(ints, [1]) == {"среднее": 1.0, "предыдущее": 4.0, "интерполяция": 0.0}
print("Задача 12: ок")

---

## Подведите итог

Ответьте в ячейке ниже текстом, по два-три предложения на вопрос.

1. Почему `titanic.dropna()` оставляет всего 182 строки из 891 и чем это опасно для
   выводов о пассажирах?
2. Вы изучаете, как выживаемость зависела от возраста. Какую стратегию для `age` выберете
   и почему? Что не годится и почему?
3. Чем дубликаты в `registrations.csv` отличаются от 107 «дубликатов» в `titanic.csv`?

*Ваши ответы:*

1.
2.
3.